# 02 Model experimentation and comparison

Five models were trained and compared: LightGBM, Random Forest (RFC), Deep MLP, Logistic Regression and XGBoost. CV numbers come from 5-fold GroupKFold by trip on the training trips. Student-test and evaluator-test Top-1 are scored for every model against one shared per-point truth (route arc aligned with the same labeller).

In [1]:
from pathlib import Path
import json, sys, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve().parents[1]          # report/notebooks -> repository root
sys.path.insert(0, str(ROOT / "src"))
FIG, TAB = ROOT / "report" / "figures", ROOT / "report" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
RES = ROOT / "results"
DATA = {"train": ROOT / "student_dataset" / "train", "student_test": ROOT / "student_dataset" / "student_test",
        "evaluator_test": ROOT / "evaluator_test"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold"})
PAL = {"train": "#2b6cb0", "student_test": "#dd6b20", "evaluator_test": "#2f855a"}

def save(name):
    plt.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.show()

def read_whitespace(path, dtype=float):
    return pd.read_csv(path, sep=r"\s+", header=None, dtype=dtype).to_numpy()

def load_trip(part, tid):
    d = DATA[part] / tid
    return {"track": read_whitespace(d / f"{tid}.track"), "nodes": read_whitespace(d / f"{tid}.nodes"),
            "arcs": read_whitespace(d / f"{tid}.arcs", dtype=np.int64),
            "route": read_whitespace(d / f"{tid}.route", dtype=np.int64).ravel()}

def trips(part):
    return sorted(p.name for p in DATA[part].iterdir() if p.is_dir() and not p.name.startswith("."))

In [2]:
git = lambda p: json.load(open(p))
oth = git(RES / "other_models" / "metrics.json"); folds_main = pd.read_csv(RES / "cv_folds.csv"); folds_oth = pd.read_csv(RES / "other_models" / "cv_folds.csv")
st = pd.read_csv(RES / "student_test" / "student_test_summary.csv"); ev = pd.read_csv(RES / "evaluator_test" / "evaluator_test_summary.csv")
KEY = {"LightGBM": "lightgbm", "Logistic Regression": "logreg", "Random Forest": "rf", "XGBoost": "xgboost", "Deep MLP": "mlp"}
tim = {k: git(ROOT / "artifacts" / "student_test_preds" / f"{v}_time.json") for k, v in KEY.items()}
rf_t = git(ROOT / "random_forest" / "outputs" / "baseline_metrics.json")
xg_t = git(ROOT / "xg boost" / "results" / "xgb_tuned.json"); xg_b = git(ROOT / "xg boost" / "results" / "xgb_baseline.json")
ml = git(ROOT / "mlp" / "results" / "mlp_complete_report_metrics.json"); ml_cv = pd.read_csv(ROOT / "mlp" / "results" / "mlp_5fold_cv_results.csv")
ov = pd.read_csv(RES / "mlp_overfit_check" / "mlp_overfit_check.csv"); mlp_clean_cv = ov[ov.run == "clean 5-fold CV"].top1
fold_rf = [0.8237, 0.7682, 0.9524, 0.9817, 0.7553]       # per-fold Top-1 from the RFC's baseline_metrics README table
def shared(df, label, stage="classifier only"): return df[(df.model == label) & (df.stage == stage)].iloc[0]
def fs(df, name, col): v = df[df.model == name][col]; return v.mean(), v.std()
cfg = {"LightGBM": ("Boosting", "400 trees, lr 0.05, 63 leaves, 36 features"),
       "Random Forest": ("Ensemble (bagging)", "150 trees, depth 15, balanced weights, 9 features"),
       "Deep MLP": ("Neural network", "128/64 hidden units, dropout 0.10, 24 features, K=5"),
       "Logistic Regression": ("Linear", "C=1.0, standardised, 36 features"),
       "XGBoost": ("Boosting", "487 trees, depth 8, lr 0.023, 15 features")}
cvv = {"LightGBM": (*fs(folds_main, "lightgbm", "top1_acc"), fs(folds_main, "lightgbm", "cand_macro_f1")[0], fs(folds_main, "lightgbm", "log_loss")[0]),
       "Logistic Regression": (oth["logreg"]["cv_top1_acc_mean"], oth["logreg"]["cv_top1_acc_std"], oth["logreg"]["cv_cand_macro_f1_mean"], oth["logreg"]["cv_log_loss_mean"]),
       "Random Forest": (np.mean(fold_rf), np.std(fold_rf, ddof=1), rf_t["cv_macro_f1"], rf_t["cv_log_loss"]),
       "XGBoost": (xg_t["cv_summary"]["top1_acc"]["mean"], xg_t["cv_summary"]["top1_acc"]["std"], xg_t["cv_summary"]["f1_macro"]["mean"], xg_t["cv_summary"]["log_loss"]["mean"]),
       "Deep MLP": (mlp_clean_cv.mean(), mlp_clean_cv.std(), ml["cv_5fold_mean_macro_f1"], ml_cv.cv_log_loss.mean())}
train = {"LightGBM": oth["lightgbm"]["train_top1"], "Logistic Regression": oth["logreg"]["train_top1"], "Random Forest": np.nan,
         "XGBoost": float(np.mean([f["train_top1_acc"] for f in xg_b["cv_folds"]])), "Deep MLP": float(ov[ov.partition == "fit trips (72)"].top1.mean())}
rows = []
for name in KEY:
    s, e = shared(st, name), shared(ev, name)
    rows.append(dict(model=name, key=KEY[name], family=cfg[name][0], config=cfg[name][1], cv_top1=cvv[name][0], cv_top1_sd=cvv[name][1], cv_macro_f1=cvv[name][2], cv_logloss=cvv[name][3],
        train_top1=train[name], test_top1=s.top1_acc, test_macro_f1=s.macro_f1, test_logloss=s.log_loss, test_continuity=s.route_continuity, test_miss=s.candidate_miss_rate,
        eval_top1=e.top1_acc, eval_macro_f1=e.macro_f1, eval_logloss=e.log_loss, eval_continuity=e.route_continuity, infer_s=tim[name]["inference_s"]))
T = pd.DataFrame(rows); T.to_csv(TAB / "all_models_table.csv", index=False)
T[["model", "family", "cv_top1", "cv_top1_sd", "cv_macro_f1", "test_top1", "test_macro_f1", "test_logloss", "test_continuity", "eval_top1", "infer_s"]].round(4)

,model,family,cv_top1,cv_top1_sd,cv_macro_f1,test_top1,test_macro_f1,test_logloss,test_continuity,eval_top1,infer_s
0,LightGBM,Boosting,0.9066,0.0207,0.9446,0.9037,0.9425,0.0468,0.9514,0.9254,0.3833
1,Logistic Regression,Linear,0.8865,0.0162,0.9199,0.8715,0.9151,0.0726,0.9321,0.9068,0.0308
2,Random Forest,Ensemble (bagging),0.8563,0.1049,0.8740,0.8453,0.8882,0.2096,0.9182,0.7829,0.1261
3,XGBoost,Boosting,0.8425,0.0315,0.9132,0.8489,0.9138,0.0787,0.9230,0.7801,0.0913
4,Deep MLP,Neural network,0.9051,0.0394,0.9553,0.9186,0.9540,0.0764,0.9675,0.9402,0.0581


## Cross-validation comparison

In [3]:
C = {"LightGBM": "#2f855a", "Random Forest": "#2b6cb0", "Deep MLP": "#c05621", "Logistic Regression": "#805ad5", "XGBoost": "#b7791f"}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
Tm = T.sort_values("cv_macro_f1"); ax[0].barh(Tm.model, Tm.cv_macro_f1, color=[C[m] for m in Tm.model]); ax[0].set(xlabel="5-fold CV macro F1 (mean over folds)", title="Cross-validation macro F1"); ax[0].set_xlim(0.8, 1)
for i, v in enumerate(Tm.cv_macro_f1): ax[0].text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)
Tt = T.sort_values("cv_top1"); ax[1].barh(Tt.model, Tt.cv_top1, xerr=Tt.cv_top1_sd, color=[C[m] for m in Tt.model], capsize=3); ax[1].set(xlabel="5-fold CV Top-1 (mean, bar = std over folds)", title="Cross-validation Top-1 accuracy"); ax[1].set_xlim(0.7, 1)
for i, v in enumerate(Tt.cv_top1): ax[1].text(0.705, i, f"{v:.3f}", va="center", fontsize=9, color="white")
plt.tight_layout(); save("model_01_cv_comparison.png")

## Student-test comparison

In [4]:
Tt = T.sort_values("test_top1"); fig, ax = plt.subplots(figsize=(11, 4.4)); y = np.arange(len(Tt)); h = 0.38
ax.barh(y - h / 2, Tt.test_top1, h, label="Top-1 accuracy", color="#2b6cb0"); ax.barh(y + h / 2, Tt.test_macro_f1, h, label="macro F1", color="#dd6b20")
ax.set_yticks(y); ax.set_yticklabels(Tt.model); ax.set_xlim(0.75, 1); ax.legend(loc="lower right"); ax.set_title("Student-test (10 trips): Top-1 accuracy and macro F1")
for i, (a, b) in enumerate(zip(Tt.test_top1, Tt.test_macro_f1)): ax.text(a + 0.003, i - h / 2, f"{a:.3f}", va="center", fontsize=8); ax.text(b + 0.003, i + h / 2, f"{b:.3f}", va="center", fontsize=8)
plt.tight_layout(); save("model_02_student_test.png")
Tl = T.sort_values("test_logloss"); fig, ax = plt.subplots(figsize=(10, 3.8)); ax.barh(Tl.model, Tl.test_logloss, color=[C[m] for m in Tl.model]); ax.set(xlabel="student-test log loss (lower is better)", title="Log loss comparison")
for i, v in enumerate(Tl.test_logloss): ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); save("model_03_log_loss.png")

## Evaluator-test comparison (final check, never used for selection)

In [5]:
E = ev.copy(); E["label"] = E.model + np.where(E.stage.str.contains("HMM"), " + HMM", ""); E = E.sort_values("top1_acc")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
ax[0].barh(E.label, E.top1_acc, color=["#c05621" if "HMM" in l else "#2b6cb0" for l in E.label]); ax[0].set_xlim(0.7, 1); ax[0].set(xlabel="Top-1 accuracy", title="evaluator_test Top-1 (shared truth)")
for i, v in enumerate(E.top1_acc): ax[0].text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
ax[1].barh(E.label, E.route_continuity, color=["#c05621" if "HMM" in l else "#2b6cb0" for l in E.label]); ax[1].set_xlim(0.85, 1); ax[1].set(xlabel="route continuity", title="evaluator_test route continuity"); ax[1].set_yticklabels([])
for i, v in enumerate(E.route_continuity): ax[1].text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); save("model_04_evaluator_test.png")

## Confusion matrices (candidate level, student-test, threshold 0.5, shared truth)

In [6]:
truth = pd.read_parquet(ROOT / "artifacts" / "student_test_preds" / "truth.parquet")[["trip", "pt", "truth_arc"]]
cms = {}
for name, k in KEY.items():
    c = pd.read_parquet(ROOT / "artifacts" / "student_test_preds" / f"{k}_cand.parquet"); c["trip"] = c.trip.astype(str).str.zfill(8)
    c = c.merge(truth, on=["trip", "pt"]); y = (c.arc == c.truth_arc).to_numpy(); yhat = (c.p >= 0.5).to_numpy()
    cms[name] = np.array([[((~y) & (~yhat)).sum(), ((~y) & yhat).sum()], [(y & (~yhat)).sum(), (y & yhat).sum()]])
fig, axs = plt.subplots(2, 3, figsize=(14, 8))
for ax, (nm, cm) in zip(axs.ravel(), cms.items()):
    cmn = cm / cm.sum(1, keepdims=True)
    sns.heatmap(cmn, annot=[[f"{cmn[i, j]:.2f}\n({int(cm[i, j]):,})" for j in range(2)] for i in range(2)], fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax, xticklabels=["not true arc", "true arc"], yticklabels=["not true arc", "true arc"])
    ax.set_title(nm); ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.grid(False)
axs.ravel()[-1].axis("off"); plt.suptitle("Row-normalised confusion matrices on student_test (counts in brackets)", y=1.0)
plt.tight_layout(); save("model_05_confusion_matrices.png")
pd.DataFrame({k: dict(zip(["TN", "FP", "FN", "TP"], v.ravel())) for k, v in cms.items()}).T.to_csv(TAB / "confusion_counts.csv")

## Training and validation loss (only for models that produce epoch histories)

In [7]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for i, (nm, f) in enumerate([("untuned", "mlp_untuned_epoch_history.csv"), ("tuned", "mlp_tuned_epoch_history.csv")]):
    h = pd.read_csv(ROOT / "mlp" / "results" / f); ax[i].plot(h.train_loss, label="training loss"); ax[i].plot(h.val_loss, label="validation loss")
    ax[i].set(title=f"Deep MLP, {nm} configuration", xlabel="epoch", ylabel="hybrid BCE + listwise loss"); ax[i].legend()
lh = xg_b["loss_history"]
def curve(v):
    if isinstance(v, dict): v = list(v.values())[0]
    return list(v)
ax[2].plot(curve(lh["train"]), label="training log loss"); ax[2].plot(curve(lh["student_test"]), label="student-test log loss")
ax[2].set(title="XGBoost, boosting rounds", xlabel="boosting round", ylabel="log loss"); ax[2].legend()
plt.tight_layout(); save("model_06_training_curves.png")

## Feature importance

In [8]:
imp = pd.read_csv(RES / "lightgbm_feature_importance.csv").head(15).iloc[::-1]; rfi = pd.Series(rf_t["feature_importances"]).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].barh(imp.feature, imp.gain / imp.gain.sum(), color="#2f855a"); ax[0].set(title="LightGBM: top 15 features (share of total gain)", xlabel="share of gain")
ax[1].barh(rfi.index, rfi.values, color="#2b6cb0"); ax[1].set(title="Random Forest: impurity importance (9 features)", xlabel="importance")
plt.tight_layout(); save("model_07_feature_importance.png")

## Over and underfitting: train against CV against held-out

In [9]:
G = T.dropna(subset=["train_top1"]).copy()
fig, ax = plt.subplots(figsize=(12, 4.8)); x = np.arange(len(G)); w = 0.2
ax.bar(x - 1.5 * w, G.train_top1, w, label="train (fit trips)", color="#718096"); ax.bar(x - 0.5 * w, G.cv_top1, w, label="5-fold CV mean", color="#2b6cb0")
ax.bar(x + 0.5 * w, G.test_top1, w, label="student-test", color="#dd6b20"); ax.bar(x + 1.5 * w, G.eval_top1, w, label="evaluator-test", color="#2f855a")
ax.set_xticks(x); ax.set_xticklabels(G.model); ax.set_ylim(0.75, 1.03); ax.set_ylabel("Top-1 accuracy"); ax.legend(ncol=4, loc="upper center")
ax.set_title("Top-1 on train, CV and held-out trips (classifier only)"); plt.tight_layout(); save("model_08_fit_gap.png")
G[["model", "train_top1", "cv_top1", "test_top1", "eval_top1"]].round(4)

,model,train_top1,cv_top1,test_top1,eval_top1
0,LightGBM,0.9508,0.9066,0.9037,0.9254
1,Logistic Regression,0.8882,0.8865,0.8715,0.9068
3,XGBoost,0.8835,0.8425,0.8489,0.7801
4,Deep MLP,0.9003,0.9051,0.9186,0.9402


## Per-trip accuracy on evaluator_test

In [10]:
pt = pd.read_csv(RES / "evaluator_test" / "evaluator_test_per_trip_top1.csv", index_col=0)
fig, ax = plt.subplots(figsize=(13, 4.8)); sns.heatmap(pt.T, annot=True, fmt=".3f", cmap="YlGn", vmin=0.55, vmax=1, ax=ax, cbar_kws={"label": "Top-1"}); ax.grid(False)
ax.set_title("evaluator_test: Top-1 per trip and model (shared truth)"); ax.set_xlabel("trip"); plt.tight_layout(); save("model_09_per_trip_evaluator.png")

## Model selection
Selection uses only the data we are allowed to select on: student_test (weight 0.6) and the 5-fold CV mean (weight 0.4). evaluator_test is shown next to it as a one-time final check and is not part of the score.

In [11]:
cvm = T.set_index("model").cv_top1; rows = []
for label in KEY:
    for stage in ["classifier only", "classifier + HMM/Viterbi"]:
        a = st[(st.model == label) & (st.stage == stage)]; b = ev[(ev.model == label) & (ev.stage == stage)]
        if len(a) == 0: continue
        a, b = a.iloc[0], b.iloc[0]
        rows.append(dict(model=label, stage=stage, cv_top1=cvm[label], student_top1=a.top1_acc, evaluator_top1=b.top1_acc, student_continuity=a.route_continuity, evaluator_continuity=b.route_continuity))
R = pd.DataFrame(rows); R["selection_score"] = 0.6 * R.student_top1 + 0.4 * R.cv_top1
R = R.sort_values("selection_score", ascending=False).reset_index(drop=True); R.insert(0, "rank", R.index + 1)
(RES / "ranking").mkdir(exist_ok=True); R.to_csv(RES / "ranking" / "model_ranking_final.csv", index=False); R.to_csv(TAB / "final_ranking.csv", index=False)
R["label"] = R.model + np.where(R.stage.str.contains("HMM"), " + HMM", "")
fig, ax = plt.subplots(figsize=(11, 4.8)); r = R.iloc[::-1]
ax.barh(r.label, r.selection_score, color=["#c05621" if "HMM" in l else "#2b6cb0" for l in r.label], alpha=0.9)
ax.scatter(r.evaluator_top1, r.label, color="black", zorder=3, s=26, label="evaluator_test Top-1 (final check, not in score)")
ax.set_xlim(0.84, 0.96); ax.set(xlabel="selection score = 0.6 x student-test Top-1 + 0.4 x CV Top-1", title="Model selection and final evaluator check"); ax.legend(loc="lower right", fontsize=8)
for i, v in enumerate(r.selection_score): ax.text(v + 0.001, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); save("model_10_selection.png"); R.drop(columns="label").round(4)

,rank,model,stage,cv_top1,student_top1,evaluator_top1,student_continuity,evaluator_continuity,selection_score
0,1,LightGBM,classifier + HMM/Viterbi,0.9066,0.9317,0.9411,0.9958,0.9954,0.9217
1,2,Deep MLP,classifier + HMM/Viterbi,0.9051,0.9207,0.9438,0.9781,0.9762,0.9145
2,3,Deep MLP,classifier only,0.9051,0.9186,0.9402,0.9675,0.9689,0.9132
3,4,LightGBM,classifier only,0.9066,0.9037,0.9254,0.9514,0.9562,0.9049
4,5,XGBoost,classifier + HMM/Viterbi,0.8425,0.9210,0.9072,0.9950,0.9937,0.8896
5,6,Logistic Regression,classifier only,0.8865,0.8715,0.9068,0.9321,0.9360,0.8775
6,7,Random Forest,classifier only,0.8563,0.8453,0.7829,0.9182,0.9251,0.8497
7,8,XGBoost,classifier only,0.8425,0.8489,0.7801,0.9230,0.9291,0.8463


## Does the ranking depend on who labelled the points?
Each final pipeline is scored against three independent per-point labellers (the shared one and the labellers built into the XGBoost and Deep MLP pipelines).

In [12]:
lr = pd.read_csv(RES / "lightgbm_hmm" / "labeller_robustness.csv"); lr = lr[lr.pipeline.str.contains("HMM")]
lr["pipeline"] = lr.pipeline.str.replace("MLP + HMM", "Deep MLP + HMM")
hm = lr.melt(id_vars=["split", "pipeline"], var_name="labeller", value_name="top1"); hm["labeller"] = hm.labeller.str.replace("my labeller", "shared labeller").str.replace(" labeller", "")
hm["col"] = hm.split.str.replace("_test", "") + " / " + hm.labeller; piv = hm.pivot(index="pipeline", columns="col", values="top1")
fig, ax = plt.subplots(figsize=(11, 3.2)); sns.heatmap(piv, annot=True, fmt=".4f", cmap="YlGn", vmin=0.90, vmax=0.96, ax=ax, cbar=False); ax.grid(False)
ax.set_title("Top-1 of the three HMM pipelines under three labellers"); ax.set_xlabel(""); ax.set_ylabel(""); plt.xticks(rotation=25, ha="right")
plt.tight_layout(); save("model_11_labeller_robustness.png"); piv.round(4)

col,evaluator / MLP,evaluator / XGBoost,evaluator / shared,student / MLP,student / XGBoost,student / shared
pipeline,,,,,,
Deep MLP + HMM,0.9537,0.9505,0.9438,0.9335,0.9217,0.9207
LightGBM + HMM,0.9438,0.9428,0.9411,0.9366,0.9330,0.9317
XGBoost + HMM,0.9157,0.9185,0.9072,0.9321,0.9338,0.9210
